# Dataset for model training

## 1. **Objective**

In this notebook, the objective is to provide a complete dataset for the training of the LSTM Model. Hereafter is a description of the following columns used for the dataset.

### 1.1 Time columns

| **column name** | **Description** | **format** |
|---|-------|-------------|
| **timestamp** | Date timestamp. | pandas.datetime.Timestamp, 'YYYY-MM-DD hh:mm:ss' |
| **hour** | Value of the hour of the day between 0 and 23 to which we applied a sinus cyclic transformation | float between -1 and 1 |
| **day_of_week** | Value of the day of the week between 0 and 6 to which we applied a sinus cyclic transformation | float between -1 and 1 |
| **month** | Value of the month of the year between 1 and 12 to which we applied a sinus cyclic transformation | float between -1 and 1 |
| **is_weekend** | 1 if the corresponding timestamp is a Saturday or a Sunday else 0 | binary |
| **is_holiday** | 1 if the corresponding timestamp corresponds to a holiday date in the US in 2017 | binary |

### 1.2 Event columns

| **column name** | **Description** | **format** |
|---|-------|-------------|
| **FLATBUSH AV/PROSPECT PARK ENT** | a row of this column is 1 if there is an event occuring at the place  FLATBUSH AV/PROSPECT PARK ENT during the corresponding timestamp | binary |
| **LIVINGSTON ST/NEVINS ST** | a row of this column is 1 if there is an event occuring at the place  LIVINGSTON ST/NEVINS ST during the corresponding timestamp | binary |
| **FLATBUSH AV/BERGEN ST** | a row of this column is 1 if there is an event occuring at the place  FLATBUSH AV/BERGEN ST during the corresponding timestamp | binary |
| **CADMAN PLZ W/MONTAGUE ST** | a row of this column is 1 if there is an event occuring at the place  CADMAN PLZ W/MONTAGUE ST during the corresponding timestamp | binary |
| **FLATBUSH AV/PLAZA ST E** | a row of this column is 1 if there is an event occuring at the place  FLATBUSH AV/PLAZA ST E during the corresponding timestamp | binary |

### 1.3 Weather columns

| **column name** | **Description** | **format** |
|---|-------|-------------|
| **temperature_c** | Temperature in NYC in °C | float |
| **precipitation** | Amount of precipitation in NYC | positive float |
| **windspeed_kmh** | Windspeed in km/h | positive float |

### 1.4 Delay columns

| **column name** | **Description** | **format** |
|---|-------|-------------|
| **avg_delay** | Average delay in minutes on the B41 bus line at the corresponding timestamp | float |
| **dayly_lag** | Average delay in minutes on the B41 bus line at the corresponding timestamp one day ago | positive float |
| **weekly_lag** | Average delay in minutes on the B41 bus line at the corresponding timestamp one week ago | positive float |

## 2. **Imports**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import requests
from math import sin, cos, pi
warnings.filterwarnings('ignore')

## 3. **Utils**

In [3]:
def cycle_func(cycle_length, x):
    return sin(2*pi*x/cycle_length) + 0.5*cos(2*pi*x/cycle_length)

In [4]:
def fetch_weather(lat, lon, start='2017-06-01', end='2018-01-01'):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':   lat,
        'longitude':  lon,
        'start_date': start,
        'end_date':   end,
        'hourly':     'temperature_2m,precipitation,windspeed_10m',
        'timezone':   'America/New_York'
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()['hourly']
    return pd.DataFrame({
        'hour_key':      pd.to_datetime(data['time']),
        'temperature_c': data['temperature_2m'],
        'precipitation': data['precipitation'],
        'windspeed_kmh': data['windspeed_10m'],
    })

In [5]:
def string_to_int(s):
    return int(s)

print(string_to_int("06"))

6


## 4. **Load datasets**

In [6]:
data_b41 = pd.read_csv("../nyc_b41_30min.csv")
print(len(data_b41))
data_b41.head()

5762


,Scheduled_dt,avg_delay,median_delay,std_delay,n_obs
0,2017-05-31 23:30:00,9.940000,8.233333,4.720325,5
1,2017-06-01 00:00:00,73.181746,5.466667,312.375853,21
2,2017-06-01 00:30:00,2.930392,2.266667,4.206197,17
3,2017-06-01 01:00:00,0.852083,0.225000,1.984632,8
4,2017-06-01 01:30:00,1.695833,0.725000,3.001755,8


In [7]:
data_events = pd.read_csv("../B41_events_timeseries_30min.csv")
print(len(data_events))
data_events.head()

5904


,timestamp,FLATBUSH AV/PROSPECT PARK ENT,LIVINGSTON ST/NEVINS ST,FLATBUSH AV/BERGEN ST,CADMAN PLZ W/MONTAGUE ST,FLATBUSH AV/PLAZA ST E
0,2017-06-01 00:00:00,0,0,0,0,0
1,2017-06-01 00:30:00,0,0,0,0,0
2,2017-06-01 01:00:00,0,0,0,0,0
3,2017-06-01 01:30:00,0,0,0,0,0
4,2017-06-01 02:00:00,0,0,0,0,0


In [8]:
NYC_LAT, NYC_LON = 40.6602, -73.9496

weather = fetch_weather(NYC_LAT, NYC_LON)
weather['is_raining'] = (weather['precipitation'] > 0).astype(int)

print(f'Weather rows: {len(weather):,}')
print(f'Date range  : {weather.hour_key.min()} -> {weather.hour_key.max()}')
print(f'Rain days   : {weather.groupby(weather.hour_key.dt.date)["is_raining"].max().sum()}')
weather.head(3)

Weather rows: 5,160
Date range  : 2017-06-01 00:00:00 -> 2018-01-01 23:00:00
Rain days   : 115


,hour_key,temperature_c,precipitation,windspeed_kmh,is_raining
0,2017-06-01 00:00:00,16.6,0.0,7.2,0
1,2017-06-01 01:00:00,17.6,0.0,11.6,0
2,2017-06-01 02:00:00,17.4,0.0,11.0,0


In [9]:
weather['hour_key'] = pd.to_datetime(weather['hour_key'])
weather = weather.set_index('hour_key')
weather = weather.resample('30min').interpolate(method='linear')
weather["month"] = weather.index.month
weather = weather[weather["month"].isin([6,8,10,12])]
weather.reset_index(drop = False)
print(len(weather))
weather.head(3)

5904


,temperature_c,precipitation,windspeed_kmh,is_raining,month
hour_key,,,,,
2017-06-01 00:00:00,16.6,0.0,7.2,0.0,6
2017-06-01 00:30:00,17.1,0.0,9.4,0.0,6
2017-06-01 01:00:00,17.6,0.0,11.6,0.0,6


In [10]:
weather.iloc[len(weather)-1]

temperature_c   -12.4
precipitation     0.0
windspeed_kmh    18.6
is_raining        0.0
month            12.0
Name: 2017-12-31 23:30:00, dtype: float64

In [11]:
print(weather.columns)

Index(['temperature_c', 'precipitation', 'windspeed_kmh', 'is_raining',
       'month'],
      dtype='str')


## 5. **Make the final dataset (we lack weather & workroads though ...)**

### 5.1 Concatenate 3 datasets : weather, B41, events

In [12]:
data_final = pd.merge(data_events,data_b41, left_on='timestamp', right_on='Scheduled_dt', how = 'left')
data_final['timestamp'] = pd.to_datetime(data_final['timestamp'])
data_final = pd.merge(data_final, weather, left_on='timestamp', right_on='hour_key' , how = 'right')
print("length of the whole dataset : ",len(data_final))

length of the whole dataset :  5904


In [13]:
# Drop the useless columns
columns_to_drop = ['Scheduled_dt','is_raining', 'n_obs']
data_final = data_final.drop(columns = columns_to_drop)
print(data_final.columns)
data_final.head(3)

Index(['timestamp', 'FLATBUSH AV/PROSPECT PARK ENT', 'LIVINGSTON ST/NEVINS ST',
       'FLATBUSH AV/BERGEN ST', 'CADMAN PLZ W/MONTAGUE ST',
       'FLATBUSH AV/PLAZA ST E', 'avg_delay', 'median_delay', 'std_delay',
       'temperature_c', 'precipitation', 'windspeed_kmh', 'month'],
      dtype='str')


,timestamp,FLATBUSH AV/PROSPECT PARK ENT,LIVINGSTON ST/NEVINS ST,FLATBUSH AV/BERGEN ST,CADMAN PLZ W/MONTAGUE ST,FLATBUSH AV/PLAZA ST E,avg_delay,median_delay,std_delay,temperature_c,precipitation,windspeed_kmh,month
0,2017-06-01 00:00:00,0,0,0,0,0,73.181746,5.466667,312.375853,16.6,0.0,7.2,6
1,2017-06-01 00:30:00,0,0,0,0,0,2.930392,2.266667,4.206197,17.1,0.0,9.4,6
2,2017-06-01 01:00:00,0,0,0,0,0,0.852083,0.225000,1.984632,17.6,0.0,11.6,6


### 5.2 Drop useless columns 

In [14]:
useless = ["median_delay", "std_delay"]
data_final = data_final.drop(columns=useless)

### 5.3 Add new columns : hour, week_day, is_week_end, is_holiday

In [15]:
# Add a week_day column
data_final["week_day"] = data_final["timestamp"].dt.day_of_week
# Add an hour column 
data_final["hour"] = data_final["timestamp"].dt.hour
# Add a month column 
data_final["month"] = data_final["timestamp"].dt.month
# Add a is_weekend column
data_final["is_weekend"] = (data_final["week_day"] > 4).astype(int)

In [16]:
# Add column is_holidays 
data_final['timestamp'] = pd.to_datetime(data_final['timestamp'])

# Conditions based on the NYC 2017 calendar
# Note : August is a full month of holiday
conditions = [
    # June : Anniversary Day (8th), Clerical Day (12th), Eid (26th), and end of the academic year starting the 29th
    (data_final['timestamp'].dt.month == 6) & (data_final['timestamp'].dt.day.isin([8, 12, 26, 29, 30])),
    
    # August : Summer Break
    (data_final['timestamp'].dt.month == 8),
    
    # October : Columbus Day (9th)
    (data_final['timestamp'].dt.month == 10) & (data_final['timestamp'].dt.day == 9),
    
    # December : Winter Recess starting the 25th
    (data_final['timestamp'].dt.month == 12) & (data_final['timestamp'].dt.day >= 25)
]

data_final['is_holiday'] = np.select(conditions, [1, 1, 1, 1], default=0)
print(data_final['is_holiday'].value_counts())

is_holiday
0    3792
1    2112
Name: count, dtype: int64


### 5.4 Nan management

In [17]:
columns_with_blanks = ['avg_delay']
#  Preparation
data_final['timestamp'] = pd.to_datetime(data_final['timestamp'])
data_final = data_final.sort_values('timestamp')

# Auxiliary columns
data_final['hour_min'] = data_final['timestamp'].dt.strftime('%H:%M')
data_final['is_weekend'] = (data_final['timestamp'].dt.dayofweek >= 5).astype(int)
data_final['month'] = data_final['timestamp'].dt.month

# Case of lonely holes in the dataset 
for col in columns_with_blanks:
    data_final[col] = data_final[col].interpolate(method='linear', limit=1)

# Case of regrouped/sequential holes in the dataset
profiles = data_final.groupby(['month', 'is_weekend', 'hour_min'])[columns_with_blanks].transform('mean')

# Apply substitution
for col in columns_with_blanks:
    data_final[col] = data_final[col].fillna(profiles[col])

# Clean the auxiliary columns
data_final = data_final.drop(columns=['hour_min'])

# Case of missing days in the end of august
mask = (data_final["timestamp"] < pd.to_datetime("2017-08-29 20:00:00")) | \
       (data_final["timestamp"] > pd.to_datetime("2017-08-31 23:30:00"))
data_final = data_final[mask]

print("Imputation terminée.")


Imputation terminée.


### 5.5 Add dayly and weekly lags

In [18]:
data_final = data_final.sort_values('timestamp')

# Dayly lag
data_final['dayly_lag'] = data_final['avg_delay'].shift(48)

# Weekly lag
data_final['weekly_lag'] = data_final['avg_delay'].shift(336)

# Remove the Na rows
data_final = data_final[data_final["timestamp"].dt.day > 7]

print(set(data_final["timestamp"].dt.day.values))

{np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23), np.int32(24), np.int32(25), np.int32(26), np.int32(27), np.int32(28), np.int32(29), np.int32(30), np.int32(31)}


### 5.6 Apply cyclic function to cycles : weeks, hours, months, ...

In [19]:
print(data_final.columns) # Cyclic columns : week_day, hour, month
print(set(data_final["month"].values)) # 6, 8, 10, 12
print(set(data_final["hour"].values)) # 0, 1, ..., 23
print(set(data_final["week_day"].values)) # 0, 1, ..., 6

Index(['timestamp', 'FLATBUSH AV/PROSPECT PARK ENT', 'LIVINGSTON ST/NEVINS ST',
       'FLATBUSH AV/BERGEN ST', 'CADMAN PLZ W/MONTAGUE ST',
       'FLATBUSH AV/PLAZA ST E', 'avg_delay', 'temperature_c', 'precipitation',
       'windspeed_kmh', 'month', 'week_day', 'hour', 'is_weekend',
       'is_holiday', 'dayly_lag', 'weekly_lag'],
      dtype='str')
{np.int32(8), np.int32(10), np.int32(12), np.int32(6)}
{np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)}
{np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)}


In [20]:
# Cyclic transformations
data_final["hour"] = data_final["hour"].apply(lambda k : cycle_func(cycle_length=24,x = k))
data_final["week_day"] = data_final["week_day"].apply(lambda k : cycle_func(cycle_length=7,x = k))
data_final["month"] = data_final["month"].apply(lambda k : cycle_func(cycle_length=12,x = k))

## 6. **Export to format .csv**

In [22]:
data_final.to_csv("final_dataset.csv")